# Bucketed guideline interpolation — sentencing starting pointThis notebook builds a **bucketed interpolated model** for the drug-trafficking starting-point sentence (in months) from the quantity of each drug. The bucket boundaries are the Hong Kong sentencing-guideline thresholds; the sentence interpolation *within* each bucket is calculated from **annotated (verified) cases** rather than hard-coded guideline values.## Design decisions- **Hybrid clamp** — each bucket's prediction is data-driven but is never allowed to leave the bucket's guideline sentence range (floor/ceiling).- **Starting point only** — quantity maps to the starting-point sentence; role, factor and plea adjustments are out of scope here.- **Data split** — interpolation is fit on the train partition only and evaluated on the held-out test partition (same split as `stage_model_analysis`).## MethodFor each bucket we record the annotated training cases `(quantity, starting point)` and fit a within-bucket interpolation `t = median(t) + slope · (u − median(u))`, where `u` is the position of the quantity inside the bucket's quantity range and `t` is the position of the sentence inside the guideline sentence range. The slope is the least-squares slope clamped non-negative, pivoted at the medians for robustness, and `t` is clamped to `[0, 1]`. Open-ended buckets fit `(quantity, sentence)` directly. Buckets with fewer than `MIN_BUCKET_SUPPORT` annotated cases fall back to pure guideline interpolation (`t = u`).

In [1]:
import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from linear_interpolation_model import (
	assign_partition,
	clean_quantity,
	get_notebook_dir,
	is_inferred,
	load_documents,
	total_months,
)

MIN_BUCKET_SUPPORT = 3
RANDOM_SEED = 2026
MODEL_VERSION = '2026-08-09'
ARTIFACT_FILENAME = 'bucketed_interpolation_model.json'
REPORT_FILENAME = 'bucketed_interpolation_analysis.xlsx'
EXCEL_ILLEGAL_CHARACTERS = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

# Guideline buckets. quantity bounds (grams), sentence bounds (months).
# kinds:
#   bounded          - quantity range and sentence range both finite; clamp to range
#   discretion_start - no guideline range below the first band; clamp [0, next band floor]
#   open_up          - sentence floor only ("X months upwards"); clamp >= floor
#   discretion_top   - "at the sentencer's discretion, practical ceiling 35 years"; clamp [0, 420]
GUIDELINE_BUCKETS = {'Cocaine': [{'low_q': 0, 'high_q': 10, 'low_s': 24, 'high_s': 60, 'kind': 'bounded'},
             {'low_q': 10, 'high_q': 50, 'low_s': 60, 'high_s': 96, 'kind': 'bounded'},
             {'low_q': 50, 'high_q': 200, 'low_s': 96, 'high_s': 144, 'kind': 'bounded'},
             {'low_q': 200, 'high_q': 500, 'low_s': 144, 'high_s': 192, 'kind': 'bounded'},
             {'low_q': 500, 'high_q': 1500, 'low_s': 192, 'high_s': 240, 'kind': 'bounded'},
             {'low_q': 1500, 'high_q': 5000, 'low_s': 240, 'high_s': 288, 'kind': 'bounded'},
             {'low_q': 5000, 'high_q': 15000, 'low_s': 288, 'high_s': 324, 'kind': 'bounded'},
             {'low_q': 15000, 'high_q': 30000, 'low_s': 324, 'high_s': 360, 'kind': 'bounded'},
             {'low_q': 30000, 'high_q': None, 'low_s': 0, 'high_s': 420, 'kind': 'discretion_top'}],
 'Ketamine': [{'low_q': 0, 'high_q': 1, 'low_s': 0, 'high_s': 24, 'kind': 'discretion_start'},
              {'low_q': 1, 'high_q': 10, 'low_s': 24, 'high_s': 48, 'kind': 'bounded'},
              {'low_q': 10, 'high_q': 50, 'low_s': 48, 'high_s': 72, 'kind': 'bounded'},
              {'low_q': 50, 'high_q': 300, 'low_s': 72, 'high_s': 108, 'kind': 'bounded'},
              {'low_q': 300, 'high_q': 600, 'low_s': 108, 'high_s': 144, 'kind': 'bounded'},
              {'low_q': 600, 'high_q': 1000, 'low_s': 144, 'high_s': 168, 'kind': 'bounded'},
              {'low_q': 1000, 'high_q': 2000, 'low_s': 168, 'high_s': 216, 'kind': 'bounded'},
              {'low_q': 2000, 'high_q': 3000, 'low_s': 216, 'high_s': 240, 'kind': 'bounded'},
              {'low_q': 3000, 'high_q': None, 'low_s': 240, 'high_s': None, 'kind': 'open_up'}],
 'Methamphetamine': [{'low_q': 0, 'high_q': 10, 'low_s': 36, 'high_s': 84, 'kind': 'bounded'},
                     {'low_q': 10, 'high_q': 70, 'low_s': 84, 'high_s': 132, 'kind': 'bounded'},
                     {'low_q': 70, 'high_q': 300, 'low_s': 132, 'high_s': 180, 'kind': 'bounded'},
                     {'low_q': 300, 'high_q': 600, 'low_s': 180, 'high_s': 216, 'kind': 'bounded'},
                     {'low_q': 600, 'high_q': 1500, 'low_s': 216, 'high_s': 240, 'kind': 'bounded'},
                     {'low_q': 1500,
                      'high_q': 5000,
                      'low_s': 240,
                      'high_s': 288,
                      'kind': 'bounded'},
                     {'low_q': 5000,
                      'high_q': 15000,
                      'low_s': 288,
                      'high_s': 324,
                      'kind': 'bounded'},
                     {'low_q': 15000,
                      'high_q': 30000,
                      'low_s': 324,
                      'high_s': 360,
                      'kind': 'bounded'},
                     {'low_q': 30000,
                      'high_q': None,
                      'low_s': 0,
                      'high_s': 420,
                      'kind': 'discretion_top'}],
 'Heroin': [{'low_q': 0, 'high_q': 10, 'low_s': 24, 'high_s': 60, 'kind': 'bounded'},
            {'low_q': 10, 'high_q': 50, 'low_s': 60, 'high_s': 96, 'kind': 'bounded'},
            {'low_q': 50, 'high_q': 200, 'low_s': 96, 'high_s': 144, 'kind': 'bounded'},
            {'low_q': 200, 'high_q': 500, 'low_s': 144, 'high_s': 192, 'kind': 'bounded'},
            {'low_q': 500, 'high_q': 1500, 'low_s': 192, 'high_s': 240, 'kind': 'bounded'},
            {'low_q': 1500, 'high_q': 5000, 'low_s': 240, 'high_s': 288, 'kind': 'bounded'},
            {'low_q': 5000, 'high_q': 15000, 'low_s': 288, 'high_s': 324, 'kind': 'bounded'},
            {'low_q': 15000, 'high_q': 30000, 'low_s': 324, 'high_s': 360, 'kind': 'bounded'},
            {'low_q': 30000, 'high_q': None, 'low_s': 0, 'high_s': 420, 'kind': 'discretion_top'}],
 'Cannabis': [{'low_q': 0, 'high_q': 2000, 'low_s': 0, 'high_s': 16, 'kind': 'bounded'},
              {'low_q': 2000, 'high_q': 3000, 'low_s': 16, 'high_s': 24, 'kind': 'bounded'},
              {'low_q': 3000, 'high_q': 6000, 'low_s': 24, 'high_s': 36, 'kind': 'bounded'},
              {'low_q': 6000, 'high_q': 9000, 'low_s': 36, 'high_s': 48, 'kind': 'bounded'},
              {'low_q': 9000, 'high_q': 15000, 'low_s': 48, 'high_s': 66, 'kind': 'bounded'},
              {'low_q': 15000, 'high_q': 45000, 'low_s': 66, 'high_s': 96, 'kind': 'bounded'},
              {'low_q': 45000, 'high_q': 90000, 'low_s': 96, 'high_s': 120, 'kind': 'bounded'},
              {'low_q': 90000, 'high_q': None, 'low_s': 120, 'high_s': None, 'kind': 'open_up'}],
 'Ecstasy': [{'low_q': 0, 'high_q': 1, 'low_s': 0, 'high_s': 24, 'kind': 'discretion_start'},
             {'low_q': 1, 'high_q': 10, 'low_s': 24, 'high_s': 48, 'kind': 'bounded'},
             {'low_q': 10, 'high_q': 50, 'low_s': 48, 'high_s': 72, 'kind': 'bounded'},
             {'low_q': 50, 'high_q': 300, 'low_s': 72, 'high_s': 108, 'kind': 'bounded'},
             {'low_q': 300, 'high_q': 600, 'low_s': 108, 'high_s': 144, 'kind': 'bounded'},
             {'low_q': 600, 'high_q': 1000, 'low_s': 144, 'high_s': 168, 'kind': 'bounded'},
             {'low_q': 1000, 'high_q': 2000, 'low_s': 168, 'high_s': 216, 'kind': 'bounded'},
             {'low_q': 2000, 'high_q': 3000, 'low_s': 216, 'high_s': 240, 'kind': 'bounded'},
             {'low_q': 3000, 'high_q': None, 'low_s': 240, 'high_s': None, 'kind': 'open_up'}],
 'Nimetazepam': [{'low_q': 0, 'high_q': 1, 'low_s': 0, 'high_s': 24, 'kind': 'discretion_start'},
                 {'low_q': 1, 'high_q': 10, 'low_s': 24, 'high_s': 48, 'kind': 'bounded'},
                 {'low_q': 10, 'high_q': 50, 'low_s': 48, 'high_s': 72, 'kind': 'bounded'},
                 {'low_q': 50, 'high_q': 300, 'low_s': 72, 'high_s': 108, 'kind': 'bounded'},
                 {'low_q': 300, 'high_q': 600, 'low_s': 108, 'high_s': 144, 'kind': 'bounded'},
                 {'low_q': 600, 'high_q': 1000, 'low_s': 144, 'high_s': 168, 'kind': 'bounded'},
                 {'low_q': 1000, 'high_q': 2000, 'low_s': 168, 'high_s': 216, 'kind': 'bounded'},
                 {'low_q': 2000, 'high_q': 3000, 'low_s': 216, 'high_s': 240, 'kind': 'bounded'},
                 {'low_q': 3000, 'high_q': None, 'low_s': 240, 'high_s': None, 'kind': 'open_up'}],
 'Midazolam-powder': [{'low_q': 0,
                       'high_q': 500,
                       'low_s': 0,
                       'high_s': 6,
                       'kind': 'discretion_start'},
                      {'low_q': 500, 'high_q': 1000, 'low_s': 6, 'high_s': 12, 'kind': 'bounded'},
                      {'low_q': 1000, 'high_q': 2000, 'low_s': 12, 'high_s': 24, 'kind': 'bounded'},
                      {'low_q': 2000, 'high_q': 3000, 'low_s': 24, 'high_s': 36, 'kind': 'bounded'},
                      {'low_q': 3000, 'high_q': 6000, 'low_s': 36, 'high_s': 54, 'kind': 'bounded'},
                      {'low_q': 6000, 'high_q': 9000, 'low_s': 54, 'high_s': 72, 'kind': 'bounded'},
                      {'low_q': 9000,
                       'high_q': None,
                       'low_s': 72,
                       'high_s': None,
                       'kind': 'open_up'}],
 'Midazolam-tablet': [{'low_q': 0,
                       'high_q': 2000,
                       'low_s': 0,
                       'high_s': 6,
                       'kind': 'discretion_start'},
                      {'low_q': 2000, 'high_q': 4000, 'low_s': 6, 'high_s': 12, 'kind': 'bounded'},
                      {'low_q': 4000, 'high_q': 8000, 'low_s': 12, 'high_s': 24, 'kind': 'bounded'},
                      {'low_q': 8000,
                       'high_q': 12000,
                       'low_s': 24,
                       'high_s': 36,
                       'kind': 'bounded'},
                      {'low_q': 12000,
                       'high_q': 24000,
                       'low_s': 36,
                       'high_s': 54,
                       'kind': 'bounded'},
                      {'low_q': 24000,
                       'high_q': 36000,
                       'low_s': 54,
                       'high_s': 72,
                       'kind': 'bounded'},
                      {'low_q': 36000,
                       'high_q': None,
                       'low_s': 72,
                       'high_s': None,
                       'kind': 'open_up'}]}

# Verified drug labels -> bucket family. Fluorodeschloroketamine follows the
# Ketamine guidelines; THC/CBD follows the Cannabis / THC guidelines; Midazolam
# is stored as grams of narcotic weight, so it follows the powder guidelines.
DRUG_FAMILY_MAP = {'Cocaine': 'Cocaine',
 'Ketamine': 'Ketamine',
 'Fluorodeschloroketamine': 'Ketamine',
 'Methamphetamine': 'Methamphetamine',
 'Heroin': 'Heroin',
 'Cannabis': 'Cannabis',
 'THC/CBD': 'Cannabis',
 'Cannabis/THC': 'Cannabis',
 'Ecstasy': 'Ecstasy',
 'Nimetazepam': 'Nimetazepam',
 'Midazolam': 'Midazolam-powder'}


In [2]:
def ols_slope(x: np.ndarray, y: np.ndarray) -> float:
	"""Least-squares slope; 0 when x is degenerate. Negative slopes are not used."""
	if len(x) < 2 or np.ptp(x) == 0:
		return 0.0
	slope = np.polyfit(x, y, 1)[0]
	if not np.isfinite(slope):
		return 0.0
	return float(max(0.0, slope))



def fit_bucket(cases: list[tuple[float, float]], bucket: dict[str, Any], fallback_s: float | None = None) -> dict[str, Any]:
	"""Compute the within-bucket interpolation parameters from annotated cases."""
	low_q = float(bucket["low_q"])
	high_q = bucket["high_q"]
	low_s = float(bucket["low_s"])
	high_s = bucket["high_s"]
	kind = bucket["kind"]
	n = len(cases)
	if high_q is not None and high_s is not None:
		if n >= MIN_BUCKET_SUPPORT:
			u = (np.array([c[0] for c in cases]) - low_q) / (high_q - low_q)
			t = (np.array([c[1] for c in cases]) - low_s) / (high_s - low_s)
			median_u = float(np.median(u))
			median_t = float(np.median(t))
			slope = ols_slope(u, t)
			status = "data-driven"
		else:
			median_u, median_t, slope, status = 0.0, 0.0, 1.0, "guideline fallback"
		return {
			"mode": "u",
			"low_q": low_q, "high_q": high_q, "low_s": low_s, "high_s": high_s,
			"kind": kind, "n_cases": n, "status": status,
			"median_u": median_u, "median_t": median_t, "slope": slope,
		}
	# Open-ended quantity: fit directly on (quantity, sentence).
	if n >= MIN_BUCKET_SUPPORT:
		median_q = float(np.median([c[0] for c in cases]))
		median_s = float(np.median([c[1] for c in cases]))
		slope = ols_slope(np.array([c[0] for c in cases]), np.array([c[1] for c in cases]))
		status = "data-driven"
	else:
		median_q = low_q
		median_s = fallback_s if fallback_s is not None else low_s
		slope = 0.0
		status = "guideline fallback"
	return {
		"mode": "q",
		"low_q": low_q, "high_q": high_q, "low_s": low_s, "high_s": high_s,
		"kind": kind, "n_cases": n, "status": status,
		"median_q": median_q, "median_s": median_s, "slope": slope,
	}



def predict_with_bucket(fit: dict[str, Any], quantity: float) -> float:
	if fit["mode"] == "u":
		u = (quantity - fit["low_q"]) / (fit["high_q"] - fit["low_q"])
		t = fit["median_t"] + fit["slope"] * (u - fit["median_u"])
		t = min(1.0, max(0.0, t))
		return fit["low_s"] + t * (fit["high_s"] - fit["low_s"])
	value = fit["median_s"] + fit["slope"] * (quantity - fit["median_q"])
	if fit["high_s"] is not None:
		return min(fit["high_s"], max(fit["low_s"], value))
	return max(fit["low_s"], value)



def find_bucket(fits: list[dict[str, Any]], quantity: float) -> dict[str, Any] | None:
	for fit in fits:
		if quantity >= fit["low_q"] and (fit["high_q"] is None or quantity < fit["high_q"]):
			return fit
	return None



def fit_all_families(training_trials: pd.DataFrame) -> dict[str, list[dict[str, Any]]]:
	family_fits: dict[str, list[dict[str, Any]]] = {}
	for family, buckets in GUIDELINE_BUCKETS.items():
		bucket_fits: list[dict[str, Any]] = []
		previous_high_s: float | None = None
		for bucket in buckets:
			cases = [
				(float(row.quantity), float(row.starting_point_months))
				for row in training_trials.itertuples(index=False)
				if row.family == family
				and row.quantity >= bucket["low_q"]
				and (bucket["high_q"] is None or row.quantity < bucket["high_q"])
			]
			fallback_s = previous_high_s if bucket["kind"] == "discretion_top" else None
			bucket_fits.append(fit_bucket(cases, bucket, fallback_s))
			if bucket["high_s"] is not None:
				previous_high_s = bucket["high_s"]
		family_fits[family] = bucket_fits
	return family_fits


In [3]:
def load_trials(notebook_dir: Path) -> tuple[pd.DataFrame, list[dict[str, Any]], dict[str, Any]]:
	"""Load verified documents, flatten to single-drug starting-point trials, assign partition."""
	documents, cache_metadata = load_documents(notebook_dir, refresh_cache=False)
	rows: list[dict[str, Any]] = []
	for document in documents:
		if document.get("exclude") is True:
			continue
		judgement = document.get("judgement") or {}
		citation = judgement.get("neutral_citation") or document.get("filename")
		case_id = str(citation or document.get("source_judgement_id") or document.get("_id"))
		for trial_index, trial in enumerate((document.get("trials") or {}).get("trials") or []):
			starting = total_months(trial.get("starting_point"))
			if starting is None or is_inferred(trial.get("starting_point")):
				continue
			amounts: dict[str, float] = {}
			for drug in trial.get("drugs") or []:
				drug_type = drug.get("drug_type")
				if not drug_type:
					continue
				if drug_type == "Other":
					other = (drug.get("other_drug_type") or "").strip().lower()
					if "midazolam" not in other:
						continue
					drug_type = "Midazolam"
				quantity, invalid = clean_quantity(drug.get("quantity"))
				if invalid:
					continue
				amounts[drug_type] = amounts.get(drug_type, 0.0) + quantity
			amounts = {name: quantity for name, quantity in amounts.items() if quantity > 0}
			if len(amounts) != 1:
				continue
			drug_type, quantity = next(iter(amounts.items()))
			family = DRUG_FAMILY_MAP.get(drug_type)
			if family is None:
				continue
			rows.append({
				"case_id": case_id,
				"trial_index": trial_index,
				"neutral_citation": citation,
				"drug_type": drug_type,
				"family": family,
				"quantity": quantity,
				"starting_point_months": starting,
			})
	trials = pd.DataFrame(rows)
	trials, _split_membership = assign_partition(trials, notebook_dir)
	return trials, documents, cache_metadata






# Load the verified documents and split into train / test (same split as stage_model_analysis).


trials, documents, cache_metadata = load_trials(get_notebook_dir())


training_trials = trials.loc[trials["partition"] == "train"].copy()


test_trials = trials.loc[trials["partition"] == "test"].copy()


print(f"{len(trials)} single-drug starting-point trials "
	  f"({len(training_trials)} train, {len(test_trials)} test)")

1421 single-drug starting-point trials (1142 train, 279 test)


In [4]:
def predict_starting_point(drug_type: str, quantity: float, family_fits: dict[str, list[dict[str, Any]]]) -> dict[str, Any]:
	"""Predict the starting-point sentence (months) for one drug and quantity."""
	family = DRUG_FAMILY_MAP.get(drug_type)
	if family is None:
		return {"status": "unsupported drug", "family": None, "predicted_months": None}
	if not np.isfinite(quantity) or quantity <= 0:
		return {"status": "invalid quantity", "family": family, "predicted_months": None}
	fit = find_bucket(family_fits[family], quantity)
	if fit is None:
		return {"status": "no bucket", "family": family, "predicted_months": None}
	return {
		"status": fit["status"],
		"family": family,
		"bucket": [fit["low_q"], fit["high_q"]],
		"predicted_months": predict_with_bucket(fit, quantity),
	}






# Fit the interpolation from annotated cases (train partition only).


family_fits = fit_all_families(training_trials)





# Quick demonstration on a few (drug, quantity) inputs.


demo = [


	("Cocaine", 5), ("Cocaine", 1000), ("Cocaine", 50000),


	("Ketamine", 5), ("Ketamine", 400),


	("Methamphetamine", 10), ("Methamphetamine", 70000),


	("Heroin", 20),


	("Cannabis", 100), ("Cannabis", 120000),


	("Ecstasy", 500),


	("Midazolam", 3000),


	("Nimetazepam", 5),


]


for drug_type, quantity in demo:


	result = predict_starting_point(drug_type, quantity, family_fits)


	print(f"{drug_type:<18} {quantity:>9,.0f} g -> {result['predicted_months']:>8,.1f} months  ({result['status']})")

Cocaine                    5 g ->     41.3 months  (data-driven)
Cocaine                1,000 g ->    240.0 months  (data-driven)
Cocaine               50,000 g ->    396.0 months  (data-driven)
Ketamine                   5 g ->     32.0 months  (data-driven)
Ketamine                 400 g ->    118.7 months  (data-driven)
Methamphetamine           10 g ->     84.0 months  (data-driven)
Methamphetamine       70,000 g ->    383.5 months  (data-driven)
Heroin                    20 g ->     68.8 months  (data-driven)
Cannabis                 100 g ->      4.5 months  (data-driven)
Cannabis             120,000 g ->    120.0 months  (data-driven)
Ecstasy                  500 g ->    131.8 months  (data-driven)
Midazolam              3,000 g ->     36.0 months  (guideline fallback)
Nimetazepam                5 g ->     34.7 months  (guideline fallback)


In [5]:
def guideline_fits(family: str) -> list[dict[str, Any]]:
	"""Pure guideline interpolation fits (t = u) for one family."""
	fits: list[dict[str, Any]] = []
	previous_high_s: float | None = None
	for b in GUIDELINE_BUCKETS[family]:
		if b["high_q"] is not None and b["high_s"] is not None:
			fits.append({
				"mode": "u",
				"low_q": b["low_q"], "high_q": b["high_q"], "low_s": b["low_s"], "high_s": b["high_s"],
				"kind": b["kind"], "median_u": 0.0, "median_t": 0.0, "slope": 1.0,
			})
		else:
			fallback = previous_high_s if b["kind"] == "discretion_top" else b["low_s"]
			fits.append({
				"mode": "q",
				"low_q": b["low_q"], "high_q": b["high_q"], "low_s": b["low_s"], "high_s": b["high_s"],
				"kind": b["kind"], "median_q": b["low_q"], "median_s": fallback, "slope": 0.0,
			})
		if b["high_s"] is not None:
			previous_high_s = b["high_s"]
	return fits



def build_test_predictions(test_trials: pd.DataFrame, family_fits: dict[str, list[dict[str, Any]]]) -> pd.DataFrame:
	pred_rows: list[dict[str, Any]] = []
	for row in test_trials.itertuples(index=False):
		family = row.family
		fit = find_bucket(family_fits[family], row.quantity)
		predicted = predict_with_bucket(fit, row.quantity) if fit is not None else np.nan
		pred_rows.append({
			"case_id": row.case_id,
			"trial_index": row.trial_index,
			"neutral_citation": row.neutral_citation,
			"drug_type": row.drug_type,
			"family": row.family,
			"quantity": row.quantity,
			"actual_months": row.starting_point_months,
			"predicted_months": predicted,
			"bucket_low_q": fit["low_q"] if fit else None,
			"bucket_high_q": fit["high_q"] if fit else None,
			"bucket_status": fit["status"] if fit else "no bucket",
		})
	return pd.DataFrame(pred_rows)



def build_metrics(predicted: pd.DataFrame) -> pd.DataFrame:
	rows: list[dict[str, Any]] = []
	all_trials = len(predicted)
	covered = predicted.dropna(subset=["predicted_months"])
	rows.append({
		"stage": "Starting point",
		"all_test_trials": all_trials,
		"covered_test_trials": len(covered),
		"coverage_rate": len(covered) / all_trials if all_trials else np.nan,
		"mae_months": (covered.actual_months - covered.predicted_months).abs().mean(),
		"median_absolute_error_months": (covered.actual_months - covered.predicted_months).abs().median(),
	})
	return pd.DataFrame(rows)



def build_guideline_baseline(test_trials: pd.DataFrame) -> pd.DataFrame:
	"""Pure guideline interpolation (t = u in every bounded bucket) as a comparison."""
	rows: list[dict[str, Any]] = []
	for row in test_trials.itertuples(index=False):
		fit = find_bucket(guideline_fits(row.family), row.quantity)
		predicted = predict_with_bucket(fit, row.quantity) if fit else np.nan
		rows.append({"predicted_guideline_months": predicted})
	return pd.DataFrame(rows)


In [6]:
def excel_safe(value: Any) -> Any:
	if isinstance(value, str):
		return EXCEL_ILLEGAL_CHARACTERS.sub("", value)
	if isinstance(value, (list, tuple, dict)):
		return EXCEL_ILLEGAL_CHARACTERS.sub("", json.dumps(value, default=str))
	return value



def write_sheet(writer: pd.ExcelWriter, dataframe: pd.DataFrame, sheet_name: str) -> None:
	safe_dataframe = dataframe.copy()
	for column in safe_dataframe:
		safe_dataframe[column] = safe_dataframe[column].map(excel_safe)
	safe_dataframe.to_excel(writer, sheet_name=sheet_name, index=False)



def sample_curve(family: str, fits: list[dict[str, Any]]) -> pd.DataFrame:
	rows: list[dict[str, Any]] = []
	for fit in fits:
		if fit["mode"] == "u":
			steps = 21
			quantities = np.linspace(fit["low_q"], fit["high_q"], steps + 1)[:-1]
		else:
			high = fit["high_q"]
			quantities = [fit["low_q"], high if high is not None else fit["low_q"] * 2]
		for quantity in quantities:
			rows.append({
				"family": family,
				"quantity": float(quantity),
				"bucket_low_q": fit["low_q"],
				"bucket_high_q": fit["high_q"],
				"status": fit["status"],
				"predicted_months": predict_with_bucket(fit, float(quantity)),
			})
	rows.append({
		"family": family,
		"quantity": np.nan,
		"bucket_low_q": None,
		"bucket_high_q": None,
		"status": "endpoint",
		"predicted_months": predict_with_bucket(fits[-1], fits[-1]["low_q"]),
	})
	return pd.DataFrame(rows)



def build_artifact(
	cache_metadata: dict[str, Any],
	family_fits: dict[str, list[dict[str, Any]]],
	training_trials: pd.DataFrame,
) -> dict[str, Any]:
	def serialize_fit(fit: dict[str, Any]) -> dict[str, Any]:
		return {key: value for key, value in fit.items() if key in {
			"low_q", "high_q", "low_s", "high_s", "kind", "n_cases", "status", "mode",
			"median_u", "median_t", "median_q", "median_s", "slope",
		}}
	return {
		"model_name": "bucketed-guideline-interpolation",
		"model_version": MODEL_VERSION,
		"training": {
			"cache_created_at": cache_metadata.get("created_at"),
			"cache_document_count": cache_metadata.get("document_count"),
			"source_excluded_document_count": cache_metadata.get("source_excluded_document_count"),
			"fit_partition": "train",
			"random_seed": RANDOM_SEED,
			"minimum_bucket_support": MIN_BUCKET_SUPPORT,
			"single_drug_starting_point_training_trials": int(len(training_trials)),
			"interpolation_method": (
				"within-bucket annotated-case interpolation (median pivot + least-squares slope "
				"clamped non-negative), clamped to the guideline sentence range (hybrid clamp); "
				"guideline fallback when fewer than 3 annotated cases"
			),
		},
		"drug_family_map": DRUG_FAMILY_MAP,
		"drugs": {
			family: {
				"unit": "grams" if family != "Midazolam-tablet" else "tablets",
				"buckets": [serialize_fit(fit) for fit in fits],
			}
			for family, fits in family_fits.items()
		},
	}



def run_analysis(
	trials: pd.DataFrame | None = None,
	documents: list[dict[str, Any]] | None = None,
	cache_metadata: dict[str, Any] | None = None,
	refresh_cache: bool = False,
) -> dict[str, Any]:
	notebook_dir = get_notebook_dir()
	if trials is None:
		trials, documents, cache_metadata = load_trials(notebook_dir)
	training_trials = trials.loc[trials["partition"] == "train"].copy()
	test_trials = trials.loc[trials["partition"] == "test"].copy()
	family_fits = fit_all_families(training_trials)

	test_predictions = build_test_predictions(test_trials, family_fits)
	guideline_baseline = build_guideline_baseline(test_trials)
	comparison = test_predictions[["case_id", "trial_index", "actual_months", "predicted_months"]].merge(
		guideline_baseline,
		left_index=True,
		right_index=True,
	)
	comparison["family"] = test_predictions["family"].values
	method_rows = [
		{
			"method": "bucketed annotated-case interpolation (hybrid clamp)",
			"test_trials": int(comparison["actual_months"].notna().sum()),
			"mae_months": (comparison.actual_months - comparison.predicted_months).abs().mean(),
			"median_absolute_error_months": (comparison.actual_months - comparison.predicted_months).abs().median(),
		},
		{
			"method": "pure guideline interpolation",
			"test_trials": int(comparison["actual_months"].notna().sum()),
			"mae_months": (comparison.actual_months - comparison.predicted_guideline_months).abs().mean(),
			"median_absolute_error_months": (comparison.actual_months - comparison.predicted_guideline_months).abs().median(),
		},
	]
	method_comparison = pd.DataFrame(method_rows)

	bucket_fit_rows: list[dict[str, Any]] = []
	for family, fits in family_fits.items():
		for fit in fits:
			bucket_fit_rows.append({
				"family": family,
				**{key: value for key, value in fit.items() if key in {
					"low_q", "high_q", "low_s", "high_s", "kind", "mode", "n_cases", "status",
					"median_u", "median_t", "median_q", "median_s", "slope",
				}},
			})
	bucket_fit = pd.DataFrame(bucket_fit_rows)

	per_family_rows: list[dict[str, Any]] = []
	for family, g in comparison.dropna(subset=["predicted_months", "predicted_guideline_months"]).groupby("family"):
		per_family_rows.append({
			"family": family,
			"test_trials": len(g),
			"model_mae_months": (g.actual_months - g.predicted_months).abs().mean(),
			"model_median_absolute_error_months": (g.actual_months - g.predicted_months).abs().median(),
			"guideline_mae_months": (g.actual_months - g.predicted_guideline_months).abs().mean(),
			"guideline_median_absolute_error_months": (g.actual_months - g.predicted_guideline_months).abs().median(),
		})
	per_family = pd.DataFrame(per_family_rows)

	per_bucket = test_predictions.dropna(subset=["predicted_months"]).copy()
	per_bucket["bucket"] = per_bucket.apply(
		lambda row: f"[{row.bucket_low_q:,.0f},{row.bucket_high_q if pd.notna(row.bucket_high_q) else '∞'})",
		axis=1,
	)
	per_bucket_error_rows: list[dict[str, Any]] = []
	for (family, bucket), g in per_bucket.dropna(subset=["predicted_months"]).groupby(["family", "bucket"]):
		per_bucket_error_rows.append({
			"family": family,
			"bucket": bucket,
			"test_trials": len(g),
			"mae_months": (g.actual_months - g.predicted_months).abs().mean(),
			"median_absolute_error_months": (g.actual_months - g.predicted_months).abs().median(),
		})
	per_bucket_error = pd.DataFrame(per_bucket_error_rows)

	curves = pd.concat([sample_curve(family, fits) for family, fits in family_fits.items()], ignore_index=True)
	metrics = build_metrics(test_predictions)

	artifact = build_artifact(cache_metadata, family_fits, training_trials)
	artifact_json = json.dumps(artifact, indent=2, sort_keys=True)
	artifact_path = notebook_dir / ARTIFACT_FILENAME
	artifact_path.write_text(artifact_json)
	typescript_path = notebook_dir.parent / "featureVerification" / "src" / "lib" / ARTIFACT_FILENAME
	typescript_path.parent.mkdir(parents=True, exist_ok=True)
	typescript_path.write_text(artifact_json)

	output_path = notebook_dir / REPORT_FILENAME
	summary = pd.DataFrame([{
		"cache_document_count": cache_metadata.get("document_count"),
		"source_excluded_document_count": cache_metadata.get("source_excluded_document_count"),
		"cache_created_at": cache_metadata.get("created_at"),
		"minimum_bucket_support": MIN_BUCKET_SUPPORT,
		"random_seed": RANDOM_SEED,
		"model_version": MODEL_VERSION,
		"training_trials": int(len(training_trials)),
		"test_trials": int(len(test_trials)),
	}])
	with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
		write_sheet(writer, summary, "summary")
		write_sheet(writer, bucket_fit, "bucket fit")
		write_sheet(writer, metrics, "held-out metrics")
		write_sheet(writer, method_comparison, "method comparison")
		write_sheet(writer, per_family, "per-family metrics")
		write_sheet(writer, per_bucket_error, "per-bucket error")
		write_sheet(writer, curves, "sampled curves")
		write_sheet(writer, test_predictions, "held-out predictions")

	assert set(family_fits) == set(GUIDELINE_BUCKETS)
	assert test_predictions["predicted_months"].notna().all()
	assert (test_predictions["predicted_months"] >= 0).all()
	assert (metrics.mae_months >= 0).all()
	for family, fits in family_fits.items():
		for fit in fits:
			assert fit["status"] in {"data-driven", "guideline fallback"}
			if fit["mode"] == "u":
				assert 0.0 <= fit["low_s"] <= fit["high_s"]
			else:
				assert fit["low_s"] >= 0
	return {
		"output_path": str(output_path),
		"artifact_path": str(artifact_path),
		"metrics": metrics,
		"method_comparison": method_comparison,
		"per_family": per_family,
		"per_bucket_error": per_bucket_error,
		"bucket_fit": bucket_fit,
		"curves": curves,
		"test_predictions": test_predictions,
		"training_trials": training_trials,
		"test_trials": test_trials,
		"family_fits": family_fits,
	}


In [7]:
# Run the full pipeline: fit -> evaluate -> export (reuses the loaded trials).
results = run_analysis(trials=trials, documents=documents, cache_metadata=cache_metadata)

print("Wrote", results["output_path"])
print("Wrote", results["artifact_path"])

Wrote

 /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/bucketed_interpolation_analysis.xlsx
Wrote /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/bucketed_interpolation_model.json


In [8]:
# Held-out metrics and method comparison (vs pure guideline interpolation).
from IPython.display import display

display(results['metrics'])
display(results['method_comparison'])
display(results['per_family'])

,stage,all_test_trials,covered_test_trials,coverage_rate,mae_months,median_absolute_error_months
0,Starting point,279,279,1.0,5.15792,1.360608


,method,test_trials,mae_months,median_absolute_error_months
0,bucketed annotated-case interpolation (hybrid ...,279,5.157920,1.360608
1,pure guideline interpolation,279,6.678929,0.652000


,family,test_trials,model_mae_months,model_median_absolute_error_months,guideline_mae_months,guideline_median_absolute_error_months
0,Cannabis,16,6.266604,3.220187,3.542204,0.483900
1,Cocaine,148,6.520420,1.863265,8.374769,0.684800
2,Ecstasy,1,0.190710,0.190710,0.048000,0.048000
3,Heroin,28,6.326660,2.227587,10.867050,1.062000
4,Ketamine,35,2.253228,1.000000,1.688491,0.626667
5,Methamphetamine,50,2.311577,0.950063,4.037293,0.578783
6,Midazolam-powder,1,1.992920,1.992920,1.992920,1.992920


## Multiple drugs — starting-point strategyThe guideline model is per-family (single-drug). For a request with several drugs the backend needs a combination rule. Two candidates are compared against the annotated outcomes of all multi-drug trials:- **A — dominant = highest sentence**: predict each drug independently and take the maximum guideline starting point.- **D — notional quantity** (HK practice, selected): for each drug, take the sentence the *total* quantity would attract in that drug's family, weight it by the drug's share of the total, and sum.The notional-quantity method (D) is used by the predictor backend.

In [9]:
def load_multi_drug_trials(notebook_dir: Path) -> pd.DataFrame:
	"""Flatten trials with two or more supported drugs and an explicit starting point."""
	documents, _cache_metadata = load_documents(notebook_dir, refresh_cache=False)
	rows: list[dict[str, Any]] = []
	for document in documents:
		if document.get("exclude") is True:
			continue
		judgement = document.get("judgement") or {}
		citation = judgement.get("neutral_citation") or document.get("filename")
		case_id = str(citation or document.get("source_judgement_id") or document.get("_id"))
		for trial_index, trial in enumerate((document.get("trials") or {}).get("trials") or []):
			starting = total_months(trial.get("starting_point"))
			if starting is None or is_inferred(trial.get("starting_point")):
				continue
			amounts: dict[str, float] = {}
			for drug in trial.get("drugs") or []:
				drug_type = drug.get("drug_type")
				if not drug_type:
					continue
				if drug_type == "Other":
					other = (drug.get("other_drug_type") or "").strip().lower()
					if "midazolam" not in other:
						continue
					drug_type = "Midazolam"
				quantity, invalid = clean_quantity(drug.get("quantity"))
				if invalid:
					continue
				amounts[drug_type] = amounts.get(drug_type, 0.0) + quantity
			amounts = {name: quantity for name, quantity in amounts.items() if quantity > 0}
			if len(amounts) < 2:
				continue
			rows.append({
				"case_id": case_id,
				"trial_index": trial_index,
				"drug_types": list(amounts.keys()),
				"quantities": list(amounts.values()),
				"starting_point_months": starting,
			})
	return pd.DataFrame(rows)



def per_drug_guideline_predictions(drug_quantities: dict[str, float]) -> list[dict[str, Any]]:
	"""Guideline starting point and bucket position for each supported drug."""
	out: list[dict[str, Any]] = []
	for drug_type, quantity in drug_quantities.items():
		family = DRUG_FAMILY_MAP.get(drug_type)
		if family is None:
			continue
		fit = find_bucket(guideline_fits(family), quantity)
		if fit is None:
			continue
		u = (
			(quantity - fit["low_q"]) / (fit["high_q"] - fit["low_q"])
			if fit["mode"] == "u"
			else np.inf
		)
		out.append({
			"drug_type": drug_type,
			"family": family,
			"quantity": quantity,
			"u": u,
			"predicted_months": predict_with_bucket(fit, quantity),
		})
	return out



def notional_weighted_months(drug_quantities: dict[str, float]) -> float:
	"""HK notional-quantity method.

	For each drug, compute the sentence the *total* quantity would attract in that
	drug's family, then weight it by the drug's share of the total quantity and sum.
	"""
	total = sum(drug_quantities.values())
	if total <= 0:
		return 0.0
	contribution: float = 0.0
	for drug_type, quantity in drug_quantities.items():
		family = DRUG_FAMILY_MAP.get(drug_type)
		if family is None:
			continue
		fit = find_bucket(guideline_fits(family), total)
		if fit is None:
			continue
		contribution += predict_with_bucket(fit, total) * (quantity / total)
	return contribution



def multi_drug_strategy_comparison(trials: pd.DataFrame) -> pd.DataFrame:
	"""Compare multi-drug starting-point strategies against annotated outcomes."""
	rows: list[dict[str, Any]] = []
	for row in trials.itertuples(index=False):
		drug_quantities = dict(zip(row.drug_types, row.quantities))
		preds = per_drug_guideline_predictions(drug_quantities)
		if not preds:
			continue
		pred_df = pd.DataFrame(preds)
		rows.append({
			"case_id": row.case_id,
			"actual_months": row.starting_point_months,
			"max_sentence_months": pred_df["predicted_months"].max(),
			"notional_weighted_months": notional_weighted_months(drug_quantities),
			"n_drugs": len(preds),
		})
	return pd.DataFrame(rows)


In [10]:
# Load all multi-drug trials (no train/test split for this comparison) and compare strategies.
multi_trials = load_multi_drug_trials(get_notebook_dir())
strategy_results = multi_drug_strategy_comparison(multi_trials)
print(f"{len(multi_trials)} multi-drug trials")

strategy_rows = []
for name, column in [('A: dominant = highest sentence', 'max_sentence_months'),
                    ('D: notional quantity (weighted)', 'notional_weighted_months')]:
	errors = (strategy_results.actual_months - strategy_results[column]).abs()
	strategy_rows.append({
		'strategy': name,
		'trials': len(strategy_results),
		'mae_months': errors.mean(),
		'median_absolute_error_months': errors.median(),
	})
strategy_comparison = pd.DataFrame(strategy_rows)
display(strategy_comparison)

801 multi-drug trials


,strategy,trials,mae_months,median_absolute_error_months
0,A: dominant = highest sentence,801,8.274946,4.376000
1,D: notional quantity (weighted),801,9.306928,2.239315


## Reading the outputs- `notebooks/bucketed_interpolation_model.json` — machine-readable model artifact (also mirrored to `featureVerification/src/lib/`).- `notebooks/bucketed_interpolation_analysis.xlsx` — full report: summary, per-bucket fits, held-out metrics, method comparison (data-driven vs pure guideline), per-family and per-bucket error, sampled prediction curves, and per-case held-out predictions.`predict_starting_point(drug_type, quantity)` is the prediction API — call it with any supported drug label and quantity in grams.